# AutoSort — ACT training on Colab

Train an ACT policy on a LeRobot dataset using Colab's free GPU, then push it to the Hugging Face Hub.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**.

Colab disconnects after a while, so we checkpoint to Google Drive and can resume.

## 1. Install LeRobot

In [ ]:
!pip install -q "lerobot[feetech]" "huggingface_hub[cli]"
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Log in to Hugging Face
Paste a **WRITE** token (huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Mount Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Settings
Match these to your dataset and Hub username.

In [ ]:
HF_USER = 'Henry-Wang0225'
DATASET = 'gear_combined_v2'      # <HF_USER>/<DATASET>
JOB_NAME = 'act_' + DATASET
STEPS = 100000
OUTPUT_DIR = '/content/drive/MyDrive/autosort/train/' + JOB_NAME

## 5. Train
Add `--resume=true` to continue from the Drive checkpoint after a disconnect.

In [ ]:
!lerobot-train \
  --dataset.repo_id={HF_USER}/{DATASET} \
  --policy.type=act \
  --policy.device=cuda \
  --output_dir={OUTPUT_DIR} \
  --job_name={JOB_NAME} \
  --steps={STEPS} \
  --policy.repo_id={HF_USER}/{JOB_NAME} \
  --policy.push_to_hub=true \
  --wandb.enable=false

## 6. Deploy
The trained policy is now at `{HF_USER}/{JOB_NAME}` on the Hub. Back on the robot machine:

```bash
./scripts/rollout.sh Henry-Wang0225/act_gear_combined_v2
```